<a href="https://colab.research.google.com/github/mhkbin/AIML-Lab/blob/main/AI_Lab_3_ML_Models_Stroke.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Lab 3 - Machine Learning Models on the Processed Stroke Dataset
Five supervised models + one ANN, compared on Accuracy, Precision, Recall, F1-Score, AUC-ROC and Confusion Matrix.

Run top-to-bottom in Google Colab.

## 1. Import libraries

In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110
np.random.seed(42)
tf.random.set_seed(42)

## 2. Upload the processed dataset
Run the cell, click **Choose Files**, and select `stroke_processed_5k.csv`.

In [ ]:
from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print("Uploaded:", filename)

In [ ]:
df = pd.read_csv(filename)
print("Shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print(df["stroke"].value_counts())
df.head()

## 3. Train-test split (80:20, stratified)

In [ ]:
X = df.drop(columns=["stroke"])
y = df["stroke"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("Stroke % in train:", round(y_train.mean() * 100, 2), " in test:", round(y_test.mean() * 100, 2))

Only about 4.9% of records are stroke cases, so the data is heavily imbalanced. Models that support it use `class_weight='balanced'` so they pay attention to the rare stroke class. Because of the imbalance, **accuracy alone is misleading**. Recall, F1 and AUC matter more here.

## 4. Helper function to evaluate any model

In [ ]:
results = {}      # metrics for each model
predictions = {}  # predicted labels and probabilities for plots

def evaluate(name, y_true, y_pred, y_prob):
    results[name] = {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall":    recall_score(y_true, y_pred, zero_division=0),
        "F1-Score":  f1_score(y_true, y_pred, zero_division=0),
        "AUC-ROC":   roc_auc_score(y_true, y_prob),
    }
    predictions[name] = {"y_pred": y_pred, "y_prob": y_prob}
    print(f"--- {name} ---")
    for k, v in results[name].items():
        print(f"{k:10s}: {v:.4f}")
    print()

## 5. Five supervised models

### Model 1: Logistic Regression

In [ ]:
m = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
m.fit(X_train, y_train)
evaluate("Logistic Regression", y_test, m.predict(X_test), m.predict_proba(X_test)[:, 1])

### Model 2: K-Nearest Neighbors

In [ ]:
m = KNeighborsClassifier(n_neighbors=7)
m.fit(X_train, y_train)
evaluate("KNN", y_test, m.predict(X_test), m.predict_proba(X_test)[:, 1])

### Model 3: Decision Tree

In [ ]:
m = DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42)
m.fit(X_train, y_train)
evaluate("Decision Tree", y_test, m.predict(X_test), m.predict_proba(X_test)[:, 1])

### Model 4: Random Forest

In [ ]:
m = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight="balanced", random_state=42)
m.fit(X_train, y_train)
evaluate("Random Forest", y_test, m.predict(X_test), m.predict_proba(X_test)[:, 1])

### Model 5: Support Vector Machine (SVM)

In [ ]:
m = SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=42)
m.fit(X_train, y_train)
evaluate("SVM", y_test, m.predict(X_test), m.predict_proba(X_test)[:, 1])

## 6. Sixth model: Artificial Neural Network (ANN)

In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight("balanced", classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))
print("Class weights:", class_weights)

ann = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(16, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid"),
])
ann.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
ann.summary()

In [ ]:
early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)

history = ann.fit(
    X_train.values, y_train.values,
    validation_split=0.2, epochs=100, batch_size=32,
    class_weight=class_weights, callbacks=[early_stop], verbose=0
)
print("Training stopped after", len(history.history["loss"]), "epochs")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"], label="Train")
axes[0].plot(history.history["val_loss"], label="Validation")
axes[0].set_title("ANN Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(history.history["accuracy"], label="Train")
axes[1].plot(history.history["val_accuracy"], label="Validation")
axes[1].set_title("ANN Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
ann_prob = ann.predict(X_test.values, verbose=0).ravel()
ann_pred = (ann_prob >= 0.5).astype(int)
evaluate("ANN", y_test, ann_pred, ann_prob)

## 7. Confusion matrix of each model

In [ ]:
names = list(results.keys())
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, name in zip(axes.flatten(), names):
    cm = confusion_matrix(y_test, predictions[name]["y_pred"])
    ConfusionMatrixDisplay(cm, display_labels=["No Stroke", "Stroke"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 8. AUC-ROC curves

In [ ]:
plt.figure(figsize=(8, 6))
for name in names:
    fpr, tpr, _ = roc_curve(y_test, predictions[name]["y_prob"])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {results[name]['AUC-ROC']:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves - All Models")
plt.legend(loc="lower right")
plt.show()

## 9. Comparison of all six models

In [ ]:
results_df = pd.DataFrame(results).T.round(4)
results_df = results_df.sort_values("AUC-ROC", ascending=False)
results_df

In [ ]:
results_df.plot(kind="bar", figsize=(13, 6), colormap="viridis")
plt.title("Model Comparison - All Metrics")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=20)
plt.legend(loc="upper right", ncol=5)
plt.tight_layout()
plt.show()

## 10. Top 3 models for stroke risk prediction
Models are ranked by **AUC-ROC** (how well the model separates stroke from no-stroke across all thresholds). Recall is also shown because missing a real stroke case is the costliest error in a medical setting. Accuracy is not used for ranking because predicting 'no stroke' for everyone already gives about 95%.

In [ ]:
top3 = results_df.head(3)
print("TOP 3 MODELS (ranked by AUC-ROC)\n")
display(top3)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, name in zip(axes, top3.index):
    cm = confusion_matrix(y_test, predictions[name]["y_pred"])
    ConfusionMatrixDisplay(cm, display_labels=["No Stroke", "Stroke"]).plot(ax=ax, cmap="Greens", colorbar=False)
    ax.set_title(f"{name}\nAUC={results[name]['AUC-ROC']:.3f}  Recall={results[name]['Recall']:.3f}")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
for name in top3.index:
    fpr, tpr, _ = roc_curve(y_test, predictions[name]["y_prob"])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {results[name]['AUC-ROC']:.3f})")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curves - Top 3 Models")
plt.legend(loc="lower right")
plt.show()

best = top3.index[0]
print(f"Best model: {best}  (AUC = {top3.iloc[0]['AUC-ROC']:.3f}, Recall = {top3.iloc[0]['Recall']:.3f})")

## 11. Conclusion
In this lab, five supervised machine learning models (Logistic Regression, K-Nearest Neighbors, Decision Tree, Random Forest and Support Vector Machine) and one Artificial Neural Network (ANN) were applied to the processed stroke dataset. The models were evaluated using Accuracy, Precision, Recall, F1-Score, AUC-ROC and confusion matrices.

The top 3 models, ranked by AUC-ROC, were Logistic Regression (AUC = 0.839), the ANN (AUC = 0.831) and Random Forest (AUC = 0.826). Logistic Regression performed best overall, with a recall of 0.80, meaning it correctly identified 80% of the actual stroke cases in the test set. The ANN achieved the same recall of 0.80, and Random Forest had the highest F1-Score (0.303) and the best precision (0.207) among all models, but a lower recall of 0.56.

The dataset is highly imbalanced, with only about 4.9% stroke cases, so accuracy alone is misleading. For example, KNN reached the highest accuracy (94.9%) but had a recall and precision of 0, because it predicted “no stroke” for almost every patient. In stroke prediction, missing a real stroke case is more harmful than a false alarm, so Recall and AUC-ROC are more important than accuracy.

In conclusion, Logistic Regression is the most suitable model for stroke risk prediction on this dataset, followed by the ANN and Random Forest. The precision of all models is low, so they are best used as a screening tool to flag high-risk patients for further checks, not as a final diagnosis.
